In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
FILE_NAME = "df_of_interest.xlsx"
df = pd.read_excel(FILE_NAME, header=0)

In [3]:
df

,Invoice_No.,Invoice Date,SupplierName,Gst_No,Item_code,SubMainGroup,Invoice Qty,Standard Rate,Invoice Rate,Disc.Per,TransporterName,Destination,Amount
0,SI00000009,2014-04-01,ASIAN TUBE LTD.(UNIT-I),24AABCA2797E1ZV,PCH-20-GR2-PP,PIPE CLAMP,10.0,0.0,81.0,30.0,,NaN,567.0
1,SI00000008,2014-04-01,SIT FLEXIBLE HOSE PVT LTD,24AAKCS2010H1ZS,H--4WE-10-D-G24,H - SOL. OPE. DIRECTIONAL VALVE,2.0,0.0,5110.0,27.0,,NaN,7460.6
2,SI00000009,2014-04-01,ASIAN TUBE LTD.(UNIT-I),24AABCA2797E1ZV,"DS-1"" BSP",DOWTY SEAL,10.0,0.0,10.0,30.0,,NaN,70.0
3,SI00000009,2014-04-01,ASIAN TUBE LTD.(UNIT-I),24AABCA2797E1ZV,"DS-1/2"" BSP",DOWTY SEAL,10.0,0.0,7.0,30.0,,NaN,49.0
4,SI00000009,2014-04-01,ASIAN TUBE LTD.(UNIT-I),24AABCA2797E1ZV,"HN-1"" BSPX1"" BSP",HEX NIPPLE,10.0,0.0,94.0,30.0,,NaN,658.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
523292,SI10001350,2026-05-07,MOHIT INDUSTRIAL PRODUCTS,07CJAPS2649G1Z3,"DRV-12X1/2"" BSP",FLOW CONTROL VALVE,12.0,1270.0,1270.0,40.0,GATI CARGO,DELHI CITY,9144.0
523293,SI10001350,2026-05-07,MOHIT INDUSTRIAL PRODUCTS,07CJAPS2649G1Z3,2P-3158--C-P-S-S-B2-E,GEAR PUMP,5.0,9100.0,9100.0,30.0,GATI CARGO,DELHI CITY,31850.0
523294,SI10001346,2026-05-07,ALFA INDIA ENTERPRISE,19AIAPB3679E1ZF,JT-19--19+13.45--25+25,GEAR COUPLING WITH BORE & KEYWAY,10.0,280.0,280.0,32.0,BNL EXPRESS,KOLKATA,1904.0
523295,SI10001364,2026-05-07,FLUIDICS ENGINEERS PVT LTD,33AADCF6140J1ZT,JTS-28,GEAR COUPLING WITH BORE & KEYWAY,5.0,70.0,70.0,36.0,ORANGE CARGO CARRIERS,COIMBATORE,224.0


In [25]:
# Same column names as common/sales_register.py
GST_COL = "Gst_No"
INVOICE_DATE_COL = "Invoice Date"
AGG_COL = "Amount"
CUSTOMER_COL = "SupplierName"
FY_START_COL = "FY_Start"
FY_COL = "FY"

work = df.copy()
work[INVOICE_DATE_COL] = pd.to_datetime(work[INVOICE_DATE_COL], errors="coerce")
work[AGG_COL] = pd.to_numeric(work[AGG_COL], errors="coerce").fillna(0.0)
work[GST_COL] = work[GST_COL].astype(str).str.strip().str.upper()
work = work[work[GST_COL].notna() & (work[GST_COL] != "") & (work[GST_COL] != "NAN")]
work = work[work[INVOICE_DATE_COL].notna()]

# Indian FY (Apr–Mar), same construction as common/sales_register.load_sales_register
inv = work[INVOICE_DATE_COL]
ok = inv.notna()
y = inv.dt.year
mo = inv.dt.month
work["FY_Start"] = pd.Series(pd.NA, index=work.index, dtype="Int64")
work.loc[ok, "FY_Start"] = y[ok].to_numpy() - (mo[ok] < 4).to_numpy().astype(np.int64)
fs = work["FY_Start"]
work["FY"] = pd.Series(pd.NA, index=work.index, dtype=object)
fmask = fs.notna()
yy = fs[fmask].astype(np.int64)
work.loc[fmask, "FY"] = (
    "FY"
    + (yy % 100).astype(str).str.zfill(2)
    + "-"
    + ((yy + 1) % 100).astype(str).str.zfill(2)
)

In [27]:
def _fy_label(fy_start: int) -> str:
    yy = int(fy_start)
    return f"FY{yy % 100:02d}-{(yy + 1) % 100:02d}"


def declining_high_revenue_customers(
    d: pd.DataFrame,
    *,
    recent_fy_start: int | None = None,
    prior_fy_start: int | None = None,
    data_from_fy_start: int | None = None,
    high_revenue_pctile: float = 0.90,
    max_recent_vs_prior: float = 0.85,
    min_prior_revenue: float = 1.0,
    min_line_items_prior: int = 2,
    min_line_items_recent: int = 0,
) -> pd.DataFrame:
    """
    Indian financial year (Apr–Mar), same FY_Start / FY labels as ``load_sales_register``.

    FY x runs Apr x → Mar x+1 (label ``FY{xx}-{xx+1}`` with xx = x % 100). We bucket by
    integer ``FY_Start`` (= calendar year of April that opens the FY).

    Defaults: ``recent_fy_start`` = largest FY_Start in (optionally clipped) data;
    ``prior_fy_start`` = ``recent_fy_start - 1``.

    "High revenue": combined revenue across those two FYs >= that quantile among
    customers with revenue in either FY.

    "Declining": recent FY revenue < max_recent_vs_prior * prior FY revenue, with floors.
    """
    if FY_START_COL not in d.columns:
        raise ValueError(
            f"Expected column {FY_START_COL!r} (Indian FY). Run the prep cell above."
        )

    cols = [GST_COL, FY_START_COL, AGG_COL]
    dd = d[cols].copy()
    dd = dd[dd[FY_START_COL].notna()]
    if data_from_fy_start is not None:
        dd = dd[dd[FY_START_COL] >= int(data_from_fy_start)]
    if dd.empty:
        raise ValueError("No rows left after FY filter; check data_from_fy_start / dates.")

    fy_max = int(dd[FY_START_COL].max())
    recent_fy_start = int(recent_fy_start or fy_max)
    prior_fy_start = int(
        prior_fy_start if prior_fy_start is not None else recent_fy_start - 1
    )
    if prior_fy_start >= recent_fy_start:
        raise ValueError("prior_fy_start must be strictly less than recent_fy_start")

    m_recent = dd[FY_START_COL] == recent_fy_start
    m_prior = dd[FY_START_COL] == prior_fy_start

    def agg_window(mask: pd.Series) -> pd.DataFrame:
        g = dd.loc[mask].groupby(GST_COL, observed=True)
        return pd.DataFrame(
            {
                "revenue": g[AGG_COL].sum(),
                "lines": g.size(),
            }
        )

    R = agg_window(m_recent).rename(columns={"revenue": "recent_rev", "lines": "recent_lines"})
    P = agg_window(m_prior).rename(columns={"revenue": "prior_rev", "lines": "prior_lines"})
    out = P.join(R, how="outer").fillna(0.0)

    out["two_fy_rev"] = out["prior_rev"] + out["recent_rev"]
    active = out["two_fy_rev"] > 0
    thr = out.loc[active, "two_fy_rev"].quantile(high_revenue_pctile)
    out["high_revenue"] = out["two_fy_rev"] >= thr

    out["pct_change_vs_prior"] = np.where(
        out["prior_rev"] > 0,
        (out["recent_rev"] - out["prior_rev"]) / out["prior_rev"],
        np.nan,
    )
    out["revenue_lost_vs_prior"] = out["prior_rev"] - out["recent_rev"]

    decline_ok = (
        (out["prior_rev"] >= min_prior_revenue)
        & (out["recent_rev"] < max_recent_vs_prior * out["prior_rev"])
        & (out["prior_lines"] >= min_line_items_prior)
        & (out["recent_lines"] >= min_line_items_recent)
    )
    out["declining"] = decline_ok

    names = (
        d.groupby(GST_COL, observed=True)[CUSTOMER_COL]
        .agg(lambda s: s.mode(dropna=True).iloc[0] if len(s.mode(dropna=True)) else s.iloc[0])
        .rename("supplier")
    )
    out = out.join(names, how="left")

    out = out.reset_index().rename(columns={GST_COL: "gst"})
    out.insert(0, "prior_fy_start", prior_fy_start)
    out.insert(1, "recent_fy_start", recent_fy_start)
    out.insert(2, "prior_fy", _fy_label(prior_fy_start))
    out.insert(3, "recent_fy", _fy_label(recent_fy_start))
    return out


# --- knobs (Indian FY; FY_Start = year of April opening that FY, same as sales_register) ---
# If the export includes a partial current FY, set RECENT_FY_START to the last complete
# FY (e.g. 2025 for FY25-26 ending Mar 2026) so you compare two full Apr–Mar periods.
DATA_FROM_FY_START = 2014  # None = all FYs; else keep rows with FY_Start >= this
RECENT_FY_START = 2025  # None → latest FY_Start in filtered data
PRIOR_FY_START = None  # None → RECENT_FY_START - 1
HIGH_PCTILE = 0.90
MAX_RECENT_VS_PRIOR = 0.85
MIN_PRIOR_REVENUE = 1_000_000.0
MIN_LINES_PRIOR = 2
MIN_LINES_RECENT = 0

flags = declining_high_revenue_customers(
    work,
    recent_fy_start=RECENT_FY_START,
    prior_fy_start=PRIOR_FY_START,
    data_from_fy_start=DATA_FROM_FY_START,
    high_revenue_pctile=HIGH_PCTILE,
    max_recent_vs_prior=MAX_RECENT_VS_PRIOR,
    min_prior_revenue=MIN_PRIOR_REVENUE,
    min_line_items_prior=MIN_LINES_PRIOR,
    min_line_items_recent=MIN_LINES_RECENT,
)

watchlist = flags[flags["high_revenue"] & flags["declining"]].sort_values(
    "revenue_lost_vs_prior", ascending=False
)
watchlist

,prior_fy_start,recent_fy_start,prior_fy,recent_fy,gst,prior_rev,prior_lines,recent_rev,recent_lines,two_fy_rev,high_revenue,pct_change_vs_prior,revenue_lost_vs_prior,declining,supplier
1730,2024,2025,FY24-25,FY25-26,24AOJPV6539G1ZI,5057026.27,985.0,1033250.80,225.0,6090277.07,True,-0.795680,4023775.47,True,EXCEL ENGINEERING (SURAT)
2093,2024,2025,FY24-25,FY25-26,24CFYPP8029R1Z6,3942049.71,167.0,414250.22,60.0,4356299.93,True,-0.894915,3527799.49,True,FORCE HYDRAULICS
1368,2024,2025,FY24-25,FY25-26,24ADGFS4973B1Z4,9898975.78,2012.0,7264922.17,1441.0,17163897.95,True,-0.266094,2634053.61,True,SHINE TRADERS
2674,2024,2025,FY24-25,FY25-26,33GLXPS9167Q1ZH,2442763.50,224.0,0.00,0.0,2442763.50,True,-1.000000,2442763.50,True,DEEPAK IND HYDRAULICS
11,2024,2025,FY24-25,FY25-26,03AABCP0755B1Z4,2453701.00,220.0,194376.00,18.0,2648077.00,True,-0.920783,2259325.00,True,PRIME HYDRAULIC & PNEUMATICS (P) LTD
745,2024,2025,FY24-25,FY25-26,24AACCV7333E1ZJ,1545600.00,6.0,1.00,1.0,1545601.00,True,-0.999999,1545599.00,True,VCARE ENGINEERING PVT LTD
526,2024,2025,FY24-25,FY25-26,23AABCI2856A1Z5,3624863.40,170.0,2191663.70,127.0,5816527.10,True,-0.395380,1433199.70,True,INDOTECH INDUSTRIES (I) PVT LTD
802,2024,2025,FY24-25,FY25-26,24AADFA2612D1ZB,2798469.20,234.0,1438242.00,134.0,4236711.20,True,-0.486061,1360227.20,True,ASIAN ELECTRIC CO
2270,2024,2025,FY24-25,FY25-26,27AAACK2479C1ZP,2125520.89,51.0,896138.90,34.0,3021659.79,True,-0.578391,1229381.99,True,KIRLOSKAR PNEUMATIC COMPANY LIMITED
511,2024,2025,FY24-25,FY25-26,22AGIPN8721D1ZM,3086392.75,424.0,1978036.70,283.0,5064429.45,True,-0.359111,1108356.05,True,BIKRAM HYDRAULICS


In [24]:
df[df.SupplierName.str.contains('PRIME') & df.TransporterName.str.contains('VRL')]

,Invoice_No.,Invoice Date,SupplierName,Gst_No,Item_code,SubMainGroup,Invoice Qty,Standard Rate,Invoice Rate,Disc.Per,TransporterName,Destination,Amount
289350,SI10001907,2021-05-28,PRIME HYDRAULIC & PNEUMATICS (P) LTD,03AABCP0755B1Z4,"DRV-10X3/8"" BSP",FLOW CONTROL VALVE,20.0,975.0,975.0,35.0,VRL LOGISTICS PVT.LTD.,LUDHIYANA,12675.0
289378,SI10001907,2021-05-28,PRIME HYDRAULIC & PNEUMATICS (P) LTD,03AABCP0755B1Z4,"DRV-08X1/4"" BSP",FLOW CONTROL VALVE,10.0,875.0,875.0,35.0,VRL LOGISTICS PVT.LTD.,LUDHIYANA,5687.5
289389,SI10001907,2021-05-28,PRIME HYDRAULIC & PNEUMATICS (P) LTD,03AABCP0755B1Z4,BHA-230-101.6-145-146-90-M12,BELL HOUSING,12.0,1400.0,1400.0,35.0,VRL LOGISTICS PVT.LTD.,LUDHIYANA,10920.0
289420,SI10001907,2021-05-28,PRIME HYDRAULIC & PNEUMATICS (P) LTD,03AABCP0755B1Z4,"DRV-12X1/2"" BSP",FLOW CONTROL VALVE,20.0,1175.0,1175.0,35.0,VRL LOGISTICS PVT.LTD.,LUDHIYANA,15275.0
290271,SI10002181,2021-06-07,PRIME HYDRAULIC & PNEUMATICS (P) LTD,03AABCP0755B1Z4,1P-3072+1P-3044--C-P-S-TT-B1-N,GEAR PUMP ( MULTI COMBINATION ),1.0,9100.0,9100.0,35.0,VRL LOGISTICS PVT.LTD.,LUDHIYANA,5915.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
512005,SI10011164,2026-01-05,PRIME HYDRAULIC & PNEUMATICS (P) LTD,03AABCP0755B1Z4,BHA-130-080-106-102-45-M08,BELL HOUSING,4.0,880.0,880.0,40.0,VRL LOGISTICS LTD.,LUDHIYANA,2112.0
518626,SI10014038,2026-03-10,PRIME HYDRAULIC & PNEUMATICS (P) LTD,03AABCP0755B1Z4,"FL-1 1/2"" BSP-T-CI-U",SAE FLANGE WITH BOLT KIT,15.0,680.0,680.0,40.0,VRL LOGISTICS LTD.,LUDHIYANA,6120.0
518632,SI10014038,2026-03-10,PRIME HYDRAULIC & PNEUMATICS (P) LTD,03AABCP0755B1Z4,"DV-08X1/4"" BSP",FLOW CONTROL VALVE,25.0,790.0,790.0,40.0,VRL LOGISTICS LTD.,LUDHIYANA,11850.0
518668,SI10014038,2026-03-10,PRIME HYDRAULIC & PNEUMATICS (P) LTD,03AABCP0755B1Z4,JT-200-M,GAUGE ISOLATOR VALVE,20.0,530.0,530.0,40.0,VRL LOGISTICS LTD.,LUDHIYANA,6360.0
